### Combined Feature Table — BUTTER-E (MLP) + EC-NAS (CNN)

**This is a shared-schema *stacked* table, not a row-level merge.** BUTTER-E and EC-NAS rows are concatenated — they are never joined on any key, and there is no row-to-row correspondence between the two datasets. Each row remains distinctly attributable to its source via the `family` (`MLP`/`CNN`) and `source_dataset` (`BUTTER-E`/`EC-NAS`) columns. This table exists to support the family-holdout (RQ1) and later hardware-holdout (RQ2) experiments, where a model is trained on rows from one group and evaluated on rows from another — not to imply the two datasets describe the same underlying runs.

**Update — auxiliary columns added.** `02a_butter_e_features.ipynb` now produces 14 BUTTER-E-only auxiliary columns beyond the original 9-column core schema: `is_gpu`, 8 one-hot `shape_*` columns, and 5 dataset-*property*-based columns (`n_observations`, `n_features`, `n_classes`, `task_encoded`, `imbalance` — joined from `pmlb.csv`, replacing an earlier one-hot `dataset_*` identity encoding for generalization to unseen datasets). (`memory_fit_ratio` was tried and dropped earlier — confirmed not to help in `03a_within_butter_e.ipynb`.) None of these exist for EC-NAS. Per the thesis's Section 3.3.2 design, they're masked to `0` for EC-NAS rows here rather than dropped or left as `NaN` — so the combined table has one consistent 23-column schema, and a pooled model always sees the same input shape regardless of which family a row came from. The masking logic below (`aux_cols = [c for c in butter.columns if c not in ecnas.columns]`) is dynamic, so it picks up whatever auxiliary columns `02a` currently produces without needing to be edited here.

**Update — `is_mlp_family` flag added, fixing a real problem the RQ1 test surfaced.** `03c_cross_architecture.ipynb`'s BUTTER-E→EC-NAS holdout run showed the zero-fill masking scheme has a failure mode: a model trained on BUTTER-E learns real dependence on `n_observations`/`is_gpu` (never 0 in real BUTTER-E data), then at test time every EC-NAS row presents those as 0 — a combination the model never saw during training, not something that reflects EC-NAS's actual architectures. Result was catastrophic absolute-scale error (R²=-8.5) despite comparatively preserved rank order (Kendall-Tau 76% of ceiling). Fix: one single `is_mlp_family` column (`1` for BUTTER-E rows, `0` for EC-NAS rows) — not a masked/zero-filled auxiliary column but a real, always-populated indicator for both families, since every current MLP-only auxiliary column shares the same applicability pattern (one flag suffices; a CNN-only auxiliary column added later would need its own analogous flag only if its applicability pattern differs). Combined table is now 24 columns.

### Combined Features — BUTTER-E (MLP) + EC-NAS (CNN)

Before pooling `butter_e_features.csv` and `ec_nas_features.csv`, check how much the two families actually overlap in scale. This matters for RQ1 (architecture-family generalization): if the held-out family's feature range falls mostly outside what the model saw in training, "generalization" is really extrapolation, and a weak result wouldn't distinguish "the model can't generalize across families" from "the model was never shown parameter counts in this range."

In [ ]:
# IMPORTS & LOAD

import pandas as pd

PROCESSED_PATH = "../../data/processed/"

butter = pd.read_csv(PROCESSED_PATH + "butter_e/butter_e_features.csv")
ecnas = pd.read_csv(PROCESSED_PATH + "ec_nas/ec_nas_features.csv")

print("BUTTER-E:", butter.shape)
print("EC-NAS:", ecnas.shape)

In [ ]:
# PARAMS DISTRIBUTION COMPARISON — range overlap between the two families

b_min, b_max, b_med = butter['params'].min(), butter['params'].max(), butter['params'].median()
e_min, e_max, e_med = ecnas['params'].min(), ecnas['params'].max(), ecnas['params'].median()

print("params comparison")
print(f"{'':12}{'min':>14}{'median':>14}{'max':>14}")
print(f"{'BUTTER-E':12}{b_min:>14,}{b_med:>14,.0f}{b_max:>14,}")
print(f"{'EC-NAS':12}{e_min:>14,}{e_med:>14,.0f}{e_max:>14,}")

overlap_lo = max(b_min, e_min)
overlap_hi = min(b_max, e_max)
overlap_width = max(0, overlap_hi - overlap_lo)
b_range = b_max - b_min
e_range = e_max - e_min

print()
print(f"overlap region: [{overlap_lo:,} , {overlap_hi:,}]")
print(f"% of BUTTER-E's range covered by overlap: {100 * overlap_width / b_range:.2f}%")
print(f"% of EC-NAS's range covered by overlap:   {100 * overlap_width / e_range:.2f}%")

b_in_overlap = ((butter['params'] >= overlap_lo) & (butter['params'] <= overlap_hi)).mean()
e_in_overlap = ((ecnas['params'] >= overlap_lo) & (ecnas['params'] <= overlap_hi)).mean()
print()
print(f"% of BUTTER-E *rows* falling inside the overlap region: {100 * b_in_overlap:.2f}%")
print(f"% of EC-NAS *rows* falling inside the overlap region:   {100 * e_in_overlap:.2f}%")

**Result:**

| | min | median | max |
|---|---:|---:|---:|
| BUTTER-E | 32 | 65,536 | 16,777,216 |
| EC-NAS | 301,074 | 3,527,472 | 31,389,066 |

Range overlap `[301,074, 16,777,216]` covers 98.2% of BUTTER-E's full range but only 53.0% of EC-NAS's (EC-NAS's upper quarter, 16.8M–31.4M params, has no BUTTER-E equivalent at all).

But range coverage isn't the same as where the *data actually sits* — most of BUTTER-E's range is sparsely populated at the small end (median 65,536 is far below its own midpoint). Looking at row counts instead: only **34.0% of BUTTER-E rows** fall inside the shared range (most BUTTER-E architectures are smaller than any EC-NAS architecture measured), while **94.7% of EC-NAS rows** do.

Practical read for RQ1: holding out CNN (EC-NAS) and testing on it means testing on a family whose typical size (median ~3.5M params) sits well inside BUTTER-E's trained range — that's genuine same-scale generalization, not extrapolation. The reverse direction (holding out MLP, testing on BUTTER-E) is messier: two-thirds of BUTTER-E rows are smaller than anything EC-NAS ever saw, so a model trained only on EC-NAS and tested on most of BUTTER-E would be extrapolating below its training range, not interpolating. Worth keeping in mind when interpreting RQ1 results in either direction — a good score holding out EC-NAS means more than a good score holding out BUTTER-E, given this asymmetry.

**Consequence for reporting RQ1 (for the results/discussion section, not this notebook):** because of this scale asymmetry, the two holdout directions are not equally fair tests and should be reported separately, not pooled into one generalization number. Train-BUTTER-E→test-EC-NAS is mostly interpolation (94.7% of EC-NAS rows fall inside BUTTER-E's trained range) — a fair test of cross-family generalization. Train-EC-NAS→test-BUTTER-E is mostly extrapolation (only 34.0% of BUTTER-E rows fall inside EC-NAS's trained range) — a harder test that conflates "can't generalize across families" with "can't extrapolate to unseen scale." A low score in that direction specifically should be discussed as a possible scale-extrapolation effect, not read on its own as cross-family generalization failure.

In [ ]:
# CONCATENATE — shared-schema stack, not a join. butter now carries auxiliary columns
# that ecnas doesn't have; mask them to 0 for EC-NAS rows (per Section 3.3.2) rather than
# dropping them or leaving NaN, so the stack has one consistent column set. is_mlp_family
# is added separately, to both families, so the model can distinguish "0 = not applicable"
# from "0 = a genuine small value" (see intro markdown — this is the 03c holdout fix).

core_cols = list(ecnas.columns)
aux_cols = [c for c in butter.columns if c not in ecnas.columns]

assert set(core_cols) <= set(butter.columns), "core schema mismatch — fix upstream before stacking"
print("core columns (shared):", core_cols)
print("auxiliary columns (BUTTER-E-only, masked to 0 for EC-NAS):", aux_cols)

ecnas_masked = ecnas.copy()
for col in aux_cols:
    ecnas_masked[col] = 0
ecnas_masked = ecnas_masked[butter.columns]  # match butter's column order

combined = pd.concat([butter, ecnas_masked], axis=0, ignore_index=True)
combined['is_mlp_family'] = (combined['family'] == 'MLP').astype(int)

combined.shape

In [ ]:
# SAVE

OUT_PATH = "../../data/processed/combined/combined_features.csv"
combined.to_csv(OUT_PATH, index=False)
OUT_PATH

In [ ]:
# SANITY CHECK

print("shape:", combined.shape)
print()
print("family value counts:\n", combined['family'].value_counts())
print()
print("source_dataset value counts:\n", combined['source_dataset'].value_counts())
print()
print("is_mlp_family by family:\n", combined.groupby('family')['is_mlp_family'].unique())
print()
print("nulls per column:\n", combined.isnull().sum())